# Activity 07 — Collect airport arrivals (AeroDataBox / RapidAPI)

Coursework notebook: pull arrivals for ICAO `LIRF` (Rome Fiumicino) and normalize into a DataFrame.

**Author:** José Alberto Rocha Munguía (`@author josea`)

Setup: copy `keys.example.py` → `keys.py` and set `flights_key`.

⚠️ RapidAPI free tiers are limited (original note: max ~200 calls/month) — do not spam the endpoint.


In [ ]:
import requests
import json
from keys import *  # flights_key


In [ ]:
# Be careful not to run too often (API monthly quota).
url = (
    "https://aerodatabox.p.rapidapi.com/flights/airports/icao/LIRF/"
    "2025-09-01T08:00/2025-09-01T20:00"
)

querystring = {
    "withLeg": "true",
    "direction": "Arrival",
    "withCancelled": "true",
    "withCodeshared": "true",
    "withCargo": "true",
    "withPrivate": "true",
    "withLocation": "false",
}

headers = {
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
    "x-rapidapi-key": flights_key,
}

response = requests.request("GET", url, headers=headers, params=querystring)
print("status:", response.status_code)


In [ ]:
# Append successful payloads into a local data.json cache
if response.status_code == 200:
    new_data = response.json()
    try:
        with open("data.json", "r", encoding="utf-8") as json_file:
            existing_data = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        existing_data = []
    existing_data.append(new_data)
    with open("data.json", "w", encoding="utf-8") as json_file:
        json.dump(existing_data, json_file, indent=4)
    print("Data appended to data.json file.")
else:
    print("Failed to retrieve data from the API. Status code:", response.status_code)


In [ ]:
import pandas as pd

flight_arrivals = response.json()

def flight_extraction(flights):
    """Flatten one arrival record into a single-row DataFrame."""
    flight_data = {
        "scheduled_arrival_time": flights["arrival"]["scheduledTime"],
        "flight_number": flights["number"],
        "from": flights["departure"]["airport"]["name"],
        "airline": flights["airline"]["name"],
        "aircraft": flights.get("aircraft", {}).get("model", "No disponible"),
    }
    return pd.json_normalize(flight_data)

flight_arrivals_df = pd.concat(
    [flight_extraction(flight) for flight in flight_arrivals.get("arrivals", [])],
    ignore_index=True,
)
flight_arrivals_df.head()
